In [ ]:
import math
from statistics import mean, stdev
from scipy.stats import t, bootstrap
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
def Fasta2Dict(filepath):
    seqs = {}
    with open(filepath, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(len(lines)):
        if lines[i].startswith(">"):
            key = lines[i][1:]
            seqs[key] = lines[i+ 1]
    headers=dict(zip(seqs.values(),seqs.keys()))
    return seqs,headers

In [ ]:
Reads,headers = Fasta2Dict("READS.fasta")
reads = list(Reads.values())
Query_dict,Query_header = Fasta2Dict("QUERY.fasta")
Query = list(Query_dict.values())[0]
print(Query)

In [ ]:
Query_header

In [ ]:
def prefix(s,len=5,n=0):
    '''
    Return the beginning of len elements of a string
    '''
    return s[n:len+n]
def suffix(s,len=5,n=0):
    '''
    Return the end of len elements of a string
    '''
    start=-(len+n)
    if n==0:
        return s[start:]
    else:
        return s[start:-n]
    
def RvsComp(s):
    '''
    Return the reverse complement of the string
    '''
    s=s.upper()
    pairs={"A":"T","T":"A","G":"C","C":"G"}
    rvs = s[::-1]
    newseq=""
    for n in rvs:
        newseq=newseq+pairs[n]
    return newseq

def getKmers(reads,len_kmer):
    '''Given reads generate kmers for the ends'''
    kmers={}
    for r in reads:
        fwdKmers=[prefix(r,len_kmer),suffix(r,len_kmer)]
        #check if kmer is in dictionary if not add it
        for k in fwdKmers:
            if kmers.get(k,None)==None:
                kmers[k]=[r]
            else:
                kmers[k].append(r)
        #Get Rvs Comp kmers
        rc = RvsComp(r)
        bkwdkmers=[prefix(rc,len_kmer),suffix(rc,len_kmer)]
        for k in bkwdkmers:
            if kmers.get(k,None)==None:
                kmers[k]=[rc]
            else:
                kmers[k].append(rc)
    return kmers

#Future Experiment test multiple windows of kmers
def get_NKmers_OneTable(reads,len_kmer,n):
    '''Get kmers for n steps. For example n of 5 would get a 5 window kmer Create a lookup table with all window kmers'''
    kmers={}
    for win in range(n+1):
        for r in reads:
            fwdKmers=[prefix(r,len_kmer,win),suffix(r,len_kmer,win)]
            #check if kmer is in dictionary if not add it
            for k in fwdKmers:
                if kmers.get(k,None)==None:
                    kmers[k]=[r]
                else:
                    kmers[k].append(r)
            #Get Rvs Comp kmers
            rc = RvsComp(r)
            bkwdkmers=[prefix(rc,len_kmer,win),suffix(rc,len_kmer,win)]
            for k in bkwdkmers:
                if kmers.get(k,None)==None:
                    kmers[k]=[rc]
                else:
                    kmers[k].append(rc)
    return kmers
def get_NKmers(reads,len_kmer,n):
    '''Get kmers for n steps. For example n of 5 would get a 5 window kmer'''
    kmers={}
    win=n
    for r in reads:
        fwdKmers=[prefix(r,len_kmer,win),suffix(r,len_kmer,win)]
        #check if kmer is in dictionary if not add it
        for k in fwdKmers:
            if kmers.get(k,None)==None:
                kmers[k]=[r]
            else:
                kmers[k].append(r)
        #Get Rvs Comp kmers
        rc = RvsComp(r)
        bkwdkmers=[prefix(rc,len_kmer,win),suffix(rc,len_kmer,win)]
        for k in bkwdkmers:
            if kmers.get(k,None)==None:
                kmers[k]=[rc]
            else:
                kmers[k].append(rc)
    return kmers
    

In [ ]:
def NumReadsStats(kmers_dict):
    '''Get the mean number of reads per kmer'''
    reads=list(kmers_dict.values())
    total_reads = np.array([len(r) for r in reads])
    mean=total_reads.mean()
    std=total_reads.std(ddof=1)
    print("mean:",mean)
    boot = bootstrap((total_reads,),statistic=np.mean,n_resamples=1000,confidence_level=0.95,method="percentile",random_state=0)
    print("95% CI:",boot.confidence_interval.low, boot.confidence_interval.high)
    return {"Mean_reads_kmer": mean,"ci_lower":boot.confidence_interval.low,"ci_upper": boot.confidence_interval.high}


In [ ]:
test = reads[0]
print("Orig:", test)
print("pref ",prefix(test,6))
print("suff ",suffix(test,6))
print("pref n1",prefix(test,6,1))
print("suff n1",suffix(test,6,1))
print("pref n2",prefix(test,6,2))
print("suff n2",suffix(test,6,2))
len(test)
print("Orig:   ",test)
print("RvsOrig:",test[::-1])
print("RvsComp:",RvsComp(test))

In [ ]:
#get unique kmers
len_kmer=[3,4,5,6,7,8,9,10]
results=[]
for k in tqdm(len_kmer):
    kmers=getKmers(reads,k)
    stats=NumReadsStats(kmers)
    stats["k"] = k
    results.append(stats)

kmer_exp= pd.DataFrame(results)
print(kmer_exp)
kmer_exp.to_csv("kmer_exp.csv")

In [ ]:
kmer_exp = pd.read_csv("kmer_exp.csv")

In [ ]:
plt.errorbar(kmer_exp["k"],kmer_exp["Mean_reads_kmer"],yerr=[kmer_exp["Mean_reads_kmer"]-kmer_exp["ci_lower"],kmer_exp["ci_upper"]-kmer_exp["Mean_reads_kmer"]],fmt='o')
plt.xlabel("Kmer Length (nucleotides)")
plt.ylabel("Mean Reads Per Kmer")
plt.title("Terminal Mean Reads Per Kmer Vs. Kmer Length")
plt.show()

In [ ]:
#get unique kmers
len_kmer=[5,6,7,8]
windows=[0,1,2,3,4,5]
results=[]
for k in tqdm(len_kmer):
    for w in windows:
        kmers=get_NKmers_OneTable(reads,k,w)
        stats=NumReadsStats(kmers)
        stats["k"] = k
        stats["windows"]=w
        results.append(stats)

kmer_expWindows= pd.DataFrame(results)
print(kmer_expWindows)
#kmer_expWindows.to_csv("kmer_exp_windows.csv")

In [ ]:
#kmer_expWindows.to_csv("kmer_exp_windows.csv")
kmer_expWindows=pd.read_csv("kmer_exp_windows.csv")

In [ ]:
plt.figure(figsize=(8,6))
for w in sorted(kmer_expWindows["windows"].unique()):
    sub = kmer_expWindows[kmer_expWindows["windows"] == w].sort_values("k")
    plt.errorbar(sub["k"],sub["Mean_reads_kmer"],yerr=[sub["Mean_reads_kmer"]-sub["ci_lower"],sub["ci_upper"]-sub["Mean_reads_kmer"]],fmt='o-',capsize=4,label=f"windows={w}")
plt.xlabel("Kmer Length (nucleotides)")
plt.ylabel("Mean Reads Per Kmer")
plt.title("Mean Reads Per Kmer Vs. Kmer Length")
plt.xticks(sorted(kmer_expWindows["k"].unique()))
plt.legend(title="Window shift")
plt.show()

In [ ]:
Kmer_tables={}
len_kmer=[6,7]
windows=[0,1,2]
for k in tqdm(len_kmer):
    Kmer_tables[k]={}
    for w in windows:
        Kmer_tables[k][w]= get_NKmers(reads, k, w)

Kmer_tables

### Create mapping

In [ ]:
def createCandidate(read,seq,kmer,win,headers,strand,k_start,k_end):
    '''
    create kmer candidate, take in orig rread r, seq, kmer, window, and header dict
    Return candidate with all items stored
    readId - id of the read
    origSeq- orignal read seq
    seq- oriented sequence orig read or rvs comp
    strand- "fwd" or "rc" (reverse comp)
    kmer-kmer sequence
    win-window offset of the kmer in the origSeq
    headers -dict mapping the orignal seq to the read id
    k_start,k_end- zero based, end exclusize coords of kmer in the oriented seq
    '''
    candidate={
    "readId": headers[read],
    "origSeq": read,
    "seq": seq,
    "strand": strand,
    "kmer": kmer,
    "window": win,
    "k_start": k_start,
    "k_end": k_end,
    "read_len": len(read)
    }
    return candidate
    
def pref_coords(win,len_kmer):
    '''return start and stop of prefix coords in kmer
    zero based end exclusive'''
    s=win
    e=win+len_kmer
    return s,e
    
def suf_coords(win,len_kmer,seq):
    '''return start and stop of suffix coords in kmer
    zero based end exclusive'''
    s=len(seq)-len_kmer-win 
    e=len(seq)-win
    return s,e
    
def get_NKmers_PrefSuftables(headers,len_kmer,n):
    '''
    return prefix table, suffix table
    Create both prefix and suffix lookup tables with all window kmers
    Get kmers for n steps.
    For example, n of 5 would get a 5 window kmer and all kmers up to the 5 nucleotide 
    '''
    prefix_table={}
    suffix_table={}

    reads=headers.keys()

    for read in tqdm(reads):
        #get both orientations
        orig=read
        oriented_reads = [("fwd", read),("rc", RvsComp(read))]
        for win in range(n+1):
            #based on fwd or rvs get pref and suffix and store in lookup table
            for orientation in oriented_reads:
                fwdrvs=orientation[0]#getr orientation fwd or rc
                r=orientation[1]#get the read
                #prefix
                pref=prefix(r,len_kmer,win)
                ps,pe = pref_coords(win,len_kmer)
                pref_candidate=createCandidate(orig,r,pref,win,headers,fwdrvs,ps,pe)
                if prefix_table.get(pref,None)==None:
                    prefix_table[pref]=[pref_candidate]
                else:
                    prefix_table[pref].append(pref_candidate)
                #suffix
                suf=suffix(r,len_kmer,win)
                ss,se = suf_coords(win,len_kmer,r)
                suf_candidate=createCandidate(orig,r,suf,win,headers,fwdrvs,ss,se)
                if suffix_table.get(suf,None)==None:
                    suffix_table[suf]=[suf_candidate]
                else:
                    suffix_table[suf].append(suf_candidate)
                
    return prefix_table,suffix_table
    

In [ ]:
len_kmer=7
n=5
prefix_table,suffix_table=get_NKmers_PrefSuftables(headers,len_kmer,n)

In [ ]:
def GetMismatches(seq1, seq2):
    '''
    Return number of mismatches between two equal length strings
    Counts mismatches between strings.
    '''
    mismatches=0
    for a, b in zip(seq1, seq2):
        if a != b:
            mismatches += 1
    return mismatches

def bestseq(contig,hits,direction,len_kmer,mRate=0.1):
    '''Get best sequence from a list of candidates
    direction:
        right - means candidate extends the right side of the contig
        left - means candidate extends the left side of the contig
    Score:
    mRate-skip any candidates with a mismatch rate greater than this threshold start at 0.1
    len(overlap) * (1-mismatch_rate) + 0.5*additional sequence added
    Returns one best result dictionary, or None if no usable candidate exists.
    '''
    best = None

    for hit in hits:
        candidateSeq = hit["seq"]
        
        expectedOverlap = len_kmer+hit["window"]
        #build right
        if direction == "right":
            contig_overlap = contig[-expectedOverlap:]
            candidate_overlap = candidateSeq[:expectedOverlap]
            extension = candidateSeq[expectedOverlap:]
            new_contig = contig+extension
        #build left
        elif direction == "left":
                contig_overlap = contig[:expectedOverlap]
                candidate_overlap = candidateSeq[-expectedOverlap:]
                extension = candidateSeq[:-expectedOverlap]
                new_contig = extension+contig
        mismatches= GetMismatches(contig_overlap, candidate_overlap)
        mismatch_rate = mismatches/expectedOverlap
        #skip candidates with poor mismatch
        if mismatch_rate>mRate:
            continue

        lengthAdded=len(extension)
        #if no length is added skip
        if lengthAdded==0:
            continue

        score= expectedOverlap*(1-mismatch_rate)+0.5*lengthAdded
        result ={
                "hit": hit,
                "seq": candidateSeq,
                "direction": direction,
                "overlap_len": expectedOverlap,
                "mismatches": mismatches,
                "mismatch_rate": mismatch_rate,
                "extension": extension,
                "lengthAdded": lengthAdded,
                "score": score,
                "new_contig": new_contig
                }
        if best is None or result['score']>best["score"]:
            best=result
    return best

In [ ]:
len_kmer=7
n=5
prefix_table,suffix_table=get_NKmers_PrefSuftables(headers,len_kmer,n)

In [ ]:
def buildRight(prefix_table,Query,len_kmer,mRate=0.1,maxsteps=10000):
    '''
    return contig built right with addition history, start and stop of contig
    Using prefix table build right on the suffix of the query'''
    contig = Query
    history = []
    used_reads = set()
    #get initial coordinates for contig
    StartContig=0
    EndContig=len(Query)
    
    for step in range(maxsteps):
        if step==maxsteps:
            print("MAXSTEPS reached")
        QSuf = suffix(contig, len_kmer, 0)
        #get kmer if no kmer is found return empty list
        hits = prefix_table.get(QSuf, [])
        #Remove any reads that have already been used
        unusedHits = []
        for hit in hits:
            if hit["readId"] in used_reads:
                continue
            unusedHits.append(hit)
        #Move on if all the reads for the kmer were used and return the final contig
        if len(unusedHits) == 0:
            break
        #get the best sequence for list of hits
        best = bestseq(contig=contig,hits=unusedHits,direction="right",len_kmer=len_kmer,mRate=mRate)
        #if no best hit is found because the remaining hits have bad overlap quality, then break
        if best is None:
            break
        
        read_len=len(best["hit"]["seq"])
        #get start of the initial overlap
        qstart0=EndContig-best["overlap_len"]
        #get end
        qend0=qstart0+read_len
        best["qstart0"]=qstart0
        best["qend0"]=qend0
        #update end of contig start stays the same
        EndContig=qend0

        contig = best["new_contig"]

        used_reads.add(best["hit"]["readId"])
        history.append(best)

    return contig, history, StartContig,EndContig,used_reads

def buildLeft(suffix_table, Query, len_kmer,used_reads,mRate=0.1, maxsteps=10000):
    '''
    Take in the output of the build right contig and build left 
    return the final contig, history, and start and stop
    Using suffix table build left on the prefix of the query'''
    contig = Query
    history = []
    if used_reads is None:
        print("Warning:Build contig with right first")
        used_reads = set()
    #get initial coordinates for contig
    StartContig=0
    EndContig=len(Query) #end of right build

    for step in range(maxsteps):
        if step==maxsteps:
            print("MAXSTEPS reached")
        QPref = prefix(contig, len_kmer, 0)
        #get kmer if no kmer is found return empty list
        hits = suffix_table.get(QPref, [])
        #Remove any reads that have already been used
        unusedHits = []
        for hit in hits:
            if hit["readId"] in used_reads:
                continue
            unusedHits.append(hit)
        #Move on if all the reads for the kmer were used and return the final contig
        if len(unusedHits) == 0:
            break
        #get the best sequence for list of hits
        best = bestseq(contig=contig,hits=unusedHits,direction="left",len_kmer=len_kmer,mRate=mRate)
        #if no best hit is found because the remaining hits have bad overlap quality, then break
        if best is None:
            break

        read_len=len(best["hit"]["seq"])
        #add the overlap len to get the neg end
        qend0=StartContig+best["overlap_len"]
        #subtract length from the end to get moving edn
        qstart0=qend0-read_len

        best["qstart0"]=qstart0
        best["qend0"]=qend0

        contig=best["new_contig"]
        StartContig=qstart0

        used_reads.add(best["hit"]["readId"])
        history.append(best)
    return contig, history, StartContig,EndContig

In [ ]:
def GetFinalCoords(Rthistory,Lefthistory,start):
    '''Convert tracking zero based and exclusive coordinates into on based inclusive for the ALLELES.aln file'''
    records=Rthistory+Lefthistory
    shift=-start #get the shift from the neg start of the left buildLeft
    rows=[]
    for rec in records:
        hit=rec["hit"]
        qstart=rec["qstart0"]+shift+1
        qend=rec["qend0"]+shift
        read_len=len(hit["seq"])

        #get seq coords for if fwd or revs comp
        if hit["strand"]=="fwd":
            sstart=1
            send=read_len
        elif hit["strand"]=="rc":
            sstart=read_len
            send=1
        else:
            raise ValueError("strand must be fwd or rc")
        
        row = {
            "sseqid": hit["readId"],
            "qseqid": "contig1",
            "sstart": sstart,
            "send": send,
            "qstart": qstart,
            "qend": qend,
            "strand": hit["strand"],
            "overlap_len": rec["overlap_len"],
            "score": rec["score"]
        }
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import matplotlib.pyplot as plt

def TestOneBuilder(len_kmer, n):
    prefix_table, suffix_table = get_NKmers_PrefSuftables(headers, len_kmer, n)

    Rtcontig, Rthistory,_,_,used_reads = buildRight(prefix_table,Query,len_kmer=len_kmer,mRate=0.1,maxsteps=10000)

    Lftcontig, Lfthistory,_,_ = buildLeft(suffix_table,Rtcontig,len_kmer=len_kmer,used_reads=used_reads,mRate=0.1,maxsteps=10000)

    row = {
        "len_kmer": len_kmer,
        "n": n,
        "setting": f"k{len_kmer}_n{n}",
        "full_contig_len": len(Lftcontig),
        "right_extensions": len(Rthistory),
        "left_extensions": len(Lfthistory),
        "total_extensions": len(Rthistory) + len(Lfthistory)
    }

    return row


def TestBuilder(len_kmers, ns, n_jobs=4):
    jobs = []
    for len_kmer in len_kmers:
        for n in ns:
            jobs.append((len_kmer, n))

    rows = Parallel(n_jobs=n_jobs, verbose=10)(delayed(TestOneBuilder)(len_kmer, n)for len_kmer, n in jobs)

    results = pd.DataFrame(rows)
    results = results.sort_values(["len_kmer", "n"]).reset_index(drop=True)

    return results

len_kmers = [7,8]
ns = [4,5,6,7]

results = TestBuilder(len_kmers, ns, n_jobs=8)

results

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results["setting"], results["full_contig_len"])
plt.xlabel("Setting")
plt.ylabel("Full contig length (# nucleotides)")
plt.title("Full Contig Length by Kmer and #Windows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = results.set_index("setting")[["right_extensions", "left_extensions", "total_extensions"]]
plot_df.plot(kind="bar", figsize=(9, 5))
plt.xlabel("Setting")
plt.ylabel("Number of extensions")
plt.title("Extension Counts by Kmer and #Windows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def writeFasta(contig, output_path, header="contig1"):
    with open(output_path, "w") as f:
        f.write(f">{header}\n")
        f.write(contig + "\n")

def writeAln(aln_df, output_path):
    aln_df.to_csv(output_path, sep="\t", index=False)

In [ ]:
len_kmer=7
_, suffix_table = get_NKmers_PrefSuftables(headers, 7, 5)
prefix_table, _ = get_NKmers_PrefSuftables(headers, 7, 6)

Rtcontig, Rthistory,rtstart,rtend = buildRight(prefix_table,Query,len_kmer=len_kmer,mRate=0.1,maxsteps=10000)
Lftcontig, Lfthistory,lftstart,lftend = buildLeft(suffix_table,Rtcontig,len_kmer=len_kmer,mRate=0.1,maxsteps=10000)

In [ ]:
aln_df=GetFinalCoords(Rthistory,Lfthistory,lftstart)

In [ ]:
Fastaoutput="ALLELES.fasta"
Alnoutput="ALLELES.aln"
writeFasta(Lftcontig,Fastaoutput)
writeAln(aln_df, Alnoutput)

In [ ]:
print("Right contig length:", len(Rtcontig))
print("Left/full contig length:", len(Lftcontig))
print("Right extensions:", len(Rthistory))
print("Left extensions:", len(Lfthistory))

In [ ]:
ext_plot_df = plot_results.set_index("setting")[["right_extensions", "left_extensions", "total_extensions"]]
ext_plot_df.plot(kind="bar", figsize=(9, 5))
plt.xlabel("Setting")
plt.ylabel("Number of extensions")
plt.title("Extension Counts by Kmer and #Windows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()